In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

# Set plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10

# Safe target models directory resolution
if os.path.basename(os.getcwd()) == 'notebooks':
    models_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'models'))
else:
    models_dir = os.path.abspath(os.path.join(os.getcwd(), 'models'))

os.makedirs(models_dir, exist_ok=True)
print(f"Target Models Directory: {models_dir}")


Target Models Directory: d:\PROJECTS\HealthSence_AI\backend\models


In [2]:
# Generate Hypertension Dataset with Probabilistic Risk Modeling
np.random.seed(42)
num_records = 2500

ages = np.random.randint(18, 85, size=num_records)
genders = np.random.choice(['male', 'female', 'other'], size=num_records, p=[0.48, 0.48, 0.04])
heights = np.random.normal(170, 10, size=num_records)
weights = np.clip(heights * 0.45 + np.random.normal(0, 10, size=num_records), 40, 150)
bmis = weights / ((heights / 100) ** 2)

smokers = np.random.choice(['yes', 'no'], size=num_records, p=[0.20, 0.80])
alcohol_use = np.random.choice(['low', 'moderate', 'high'], size=num_records, p=[0.60, 0.30, 0.10])
activities = np.random.choice(['sedentary', 'moderate', 'active'], size=num_records, p=[0.35, 0.45, 0.20])
sleep_hours = np.clip(np.random.normal(7, 1.2, size=num_records), 4, 10)

systolic = np.clip(110 + (bmis - 22) * 1.2 + (ages - 30) * 0.3 + np.random.normal(0, 8, size=num_records), 85, 200).astype(int)
diastolic = np.clip(70 + (bmis - 22) * 0.8 + (ages - 30) * 0.15 + np.random.normal(0, 6, size=num_records), 55, 120).astype(int)
cholesterol = np.clip(160 + (bmis - 22) * 2.0 + (ages - 30) * 0.8 + np.random.normal(0, 15, size=num_records), 100, 380).astype(int)
glucose = np.clip(85 + (bmis - 22) * 1.5 + (ages - 30) * 0.4 + np.random.normal(0, 12, size=num_records), 55, 280).astype(int)
insulin = np.clip(5 + (glucose - 85) * 0.18 + (bmis - 22) * 0.6 + np.random.normal(0, 3, size=num_records), 2, 55).astype(int)
heart_rate = np.clip(65 + (bmis - 22) * 0.4 + np.random.normal(0, 8, size=num_records), 45, 130).astype(int)

# Target risk logit calculation for Hypertension
logit_hypertension = (
    -4.8
    + 0.055 * (systolic - 120)
    + 0.045 * (diastolic - 80)
    + 0.035 * (bmis - 24)
    + 0.025 * (ages - 35)
    + 0.015 * (heart_rate - 70)
    + 0.40 * (smokers == 'yes').astype(float)
    + 0.35 * (alcohol_use == 'high').astype(float)
    + np.random.normal(0, 0.4, size=num_records)
)
prob_hypertension = 1 / (1 + np.exp(-logit_hypertension))
y_hypertension = (np.random.rand(num_records) < prob_hypertension).astype(int)

df_hypertension = pd.DataFrame({
    'age': ages, 'gender': genders, 'height': heights, 'weight': weights, 'bmi': bmis,
    'smoking': smokers, 'alcohol': alcohol_use, 'physical_activity': activities,
    'sleep_duration': sleep_hours, 'bp_systolic': systolic, 'bp_diastolic': diastolic,
    'cholesterol': cholesterol, 'glucose': glucose, 'insulin': insulin, 'heart_rate': heart_rate,
    'hypertension': y_hypertension
})

csv_path = os.path.join(models_dir, "hypertension_dataset.csv")
df_hypertension.to_csv(csv_path, index=False)
print(f"Saved Hypertension Dataset to: {csv_path} ({len(df_hypertension)} records, {y_hypertension.sum()} positive cases [{y_hypertension.mean()*100:.1f}%])")


Saved Hypertension Dataset to: d:\PROJECTS\HealthSence_AI\backend\models\hypertension_dataset.csv (2500 records, 60 positive cases [2.4%])


In [3]:
categorical_cols = ['gender', 'smoking', 'alcohol', 'physical_activity']
numerical_cols = ['age', 'height', 'weight', 'bmi', 'sleep_duration', 'bp_systolic', 'bp_diastolic', 'cholesterol', 'glucose', 'insulin', 'heart_rate']

cat_categories = {
    'gender': ['male', 'female', 'other'],
    'smoking': ['yes', 'no'],
    'alcohol': ['low', 'moderate', 'high'],
    'physical_activity': ['sedentary', 'moderate', 'active']
}

def preprocess_features(df_input):
    X_num = df_input[numerical_cols].values
    encoded_cats = []
    for col, cats in cat_categories.items():
        for category in cats:
            encoded_cats.append((df_input[col] == category).astype(float).values)
    X_cat = np.column_stack(encoded_cats)
    return np.column_stack([X_num, X_cat])

X = preprocess_features(df_hypertension)
y = df_hypertension['hypertension'].values

# Load existing Scaler
scaler_path = os.path.join(models_dir, "scaler.pkl")
if os.path.exists(scaler_path):
    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
else:
    scaler = StandardScaler()
    scaler.fit(df_hypertension[numerical_cols].values)

# Export Feature Names
feature_names = numerical_cols.copy()
for col, cats in cat_categories.items():
    for category in cats:
        feature_names.append(f"{col}_{category}")


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[:, :11] = scaler.transform(X_train[:, :11])
X_test_scaled[:, :11] = scaler.transform(X_test[:, :11])

algorithms = {
    'logistic_regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'decision_tree': DecisionTreeClassifier(max_depth=5, min_samples_split=8, random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=120, max_depth=7, min_samples_split=6, random_state=42),
    'xgboost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.08, eval_metric='logloss', random_state=42),
    'svm': SVC(C=1.0, kernel='rbf', probability=True, random_state=42)
}

metrics_file = os.path.join(models_dir, "model_metrics.json")
metrics_report = {}
if os.path.exists(metrics_file):
    try:
        with open(metrics_file, "r") as f:
            metrics_report = json.load(f)
    except Exception:
        metrics_report = {}

disease_name = 'hypertension'
metrics_report[disease_name] = {}
trained_models = {}
eval_results = {}

print(f"\nTraining models & 5-Fold CV for target domain: {disease_name}")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for alg_name, clf in algorithms.items():
    cv_scores = cross_val_score(clf, X_train_scaled, y_train, cv=skf, scoring='accuracy')
    cv_mean = float(cv_scores.mean())
    cv_std = float(cv_scores.std())
    
    clf.fit(X_train_scaled, y_train)
    trained_models[alg_name] = clf
    
    model_filename = os.path.join(models_dir, f"{disease_name}_{alg_name}.pkl")
    with open(model_filename, "wb") as f:
        pickle.dump(clf, f)
        
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    metrics_report[disease_name][alg_name] = {
        'accuracy': round(float(acc), 4),
        'precision': round(float(prec), 4),
        'recall': round(float(rec), 4),
        'f1_score': round(float(f1), 4),
        'roc_auc': round(float(auc), 4),
        'cv_accuracy_mean': round(float(cv_mean), 4),
        'cv_accuracy_std': round(float(cv_std), 4)
    }
    print(f"  [{alg_name:<19}] Test Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f} | 5-Fold CV: {cv_mean:.4f} (+/-{cv_std:.4f})")

with open(metrics_file, "w") as f:
    json.dump(metrics_report, f, indent=2)

print(f"\nCompleted training for {disease_name}. Updated metrics written to: {metrics_file}")



Training models & 5-Fold CV for target domain: hypertension
  [logistic_regression] Test Acc: 0.9760 | F1: 0.0000 | AUC: 0.8193 | 5-Fold CV: 0.9760 (+/-0.0012)
  [decision_tree      ] Test Acc: 0.9660 | F1: 0.1053 | AUC: 0.6668 | 5-Fold CV: 0.9705 (+/-0.0043)
  [random_forest      ] Test Acc: 0.9760 | F1: 0.0000 | AUC: 0.8034 | 5-Fold CV: 0.9760 (+/-0.0012)
  [xgboost            ] Test Acc: 0.9760 | F1: 0.0000 | AUC: 0.8354 | 5-Fold CV: 0.9755 (+/-0.0010)
  [svm                ] Test Acc: 0.9760 | F1: 0.0000 | AUC: 0.7015 | 5-Fold CV: 0.9760 (+/-0.0012)

Completed training for hypertension. Updated metrics written to: d:\PROJECTS\HealthSence_AI\backend\models\model_metrics.json
